In [1]:
import torch
import os

In [2]:
import torch
from torch.utils.data import Dataset
from torchvision import datasets
from torchvision.transforms import v2
import matplotlib.pyplot as plt

# Preparing our Dataset

In [3]:

from torchvision.io import decode_image ,ImageReadMode


labels_map = {
    0: "NORMAL",
    1: "PNEUMONIA",
}   
class_to_idx = {
    "NORMAL":0,
    "PNEUMONIA":1
}

class XrayDataset(Dataset):
    def __init__(self, train_dir , transform = None , target_transform=None):
        self.img_samples = self.annotation_pair(train_dir)
        self.transform = transform
        self.target_transform = target_transform

    def annotation_pair(self,train_dir):
        samples = []
        for className in os.listdir(train_dir):
            for filename in os.listdir(os.path.join(train_dir,className)):
                samples.append((os.path.join(train_dir,className,filename),class_to_idx[className]))
        return samples  

    def __len__(self):
        return len(self.img_samples)

    def __getitem__(self,idx):
        img_path = self.img_samples[idx][0]
        label = self.img_samples[idx][1]
        image = decode_image(img_path , mode=ImageReadMode.GRAY)
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        
        return image , label
    

# Loading our Dataset

In [32]:
from torch.utils.data import DataLoader
from torchvision.transforms import v2


train_dir = "/kaggle/input/datasets/andrewmvd/pediatric-pneumonia-chest-xray/Pediatric Chest X-ray Pneumonia/train"
test_dir = "/kaggle/input/datasets/andrewmvd/pediatric-pneumonia-chest-xray/Pediatric Chest X-ray Pneumonia/test"
training_data = XrayDataset(train_dir,transform=v2.Compose([v2.ToImage(),v2.Resize((224, 224)) ,v2.ToDtype(torch.float32, scale=True)]))
test_data = XrayDataset(test_dir,transform=v2.Compose([v2.ToImage(),v2.Resize((224, 224)),v2.ToDtype(torch.float32, scale=True)]))


train_dataloader = DataLoader(training_data, batch_size=24, shuffle=True , num_workers=4, pin_memory=True)
test_dataloader = DataLoader(test_data, batch_size=24, shuffle=True, num_workers=4, pin_memory=True)




In [33]:
print(train_dataloader)
X, y = next(iter(train_dataloader))
print(X.shape)
print(y.shape)
print(f"We have {(len(train_dataloader))} batches in total" )

torch.Size([24, 1, 224, 224])
torch.Size([24])
We have 218 batches in total


# Building Our Neural Net PneumoNet V1.0

In [34]:
import torch.nn as nn
class PneumoNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.stack  = nn.Sequential(
            nn.Linear(224*224,256),
            nn.ReLU(),
            nn.Linear(256, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )
    def forward(self,x):
        x = self.flatten(x)
        logits = self.stack(x)
        return logits

model = PneumoNet().to(device)

In [35]:
total_params = sum(p.numel() for p in model.parameters())

In [36]:
print(total_params)

12853569


# Defining Hyperparameters

In [37]:
lr = 1e-2
batch_size = 24
epochs = 10

# Defining Optimizer and loss funciton

In [43]:
from torch import optim

pos_weight = torch.tensor([1349 / 3883]).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer =  optim.SGD(model.parameters(), lr=lr, weight_decay=1e-4 , , momentum=0.9)

# Building Train Loop

In [44]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

def training_loop(dataloader , model,  optimizer ,  loss_fn) : 
    size = len(dataloader.dataset)
    model.train()

    for batch , (X, y ) in enumerate(dataloader):
        X, y = X.to(device) , y.to(device)
        pred = model(X)
        loss = loss_fn(pred.squeeze(),y.float())
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        
        if batch % 10 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred.squeeze(), y.float()).item()
            correct += ((pred.squeeze() > 0).float() == y).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [45]:
for i in range(epochs):
    print(f"Epoch {i+1}\n-------------------------------")
    training_loop(train_dataloader, model, optimizer, loss_fn)
    test_loop(test_dataloader,model,loss_fn)
print("Training Complete")

Epoch 1
-------------------------------
loss: 0.129604  [   24/ 5232]
loss: 0.124290  [  264/ 5232]
loss: 0.040277  [  504/ 5232]
loss: 0.142444  [  744/ 5232]
loss: 0.081601  [  984/ 5232]
loss: 0.053230  [ 1224/ 5232]
loss: 0.032021  [ 1464/ 5232]
loss: 0.052368  [ 1704/ 5232]
loss: 0.037646  [ 1944/ 5232]
loss: 0.076882  [ 2184/ 5232]
loss: 0.043555  [ 2424/ 5232]
loss: 0.010670  [ 2664/ 5232]
loss: 0.080736  [ 2904/ 5232]
loss: 0.023375  [ 3144/ 5232]
loss: 0.026371  [ 3384/ 5232]
loss: 0.063546  [ 3624/ 5232]
loss: 0.254153  [ 3864/ 5232]
loss: 0.006687  [ 4104/ 5232]
loss: 0.112657  [ 4344/ 5232]
loss: 0.177085  [ 4584/ 5232]
loss: 0.032166  [ 4824/ 5232]
loss: 0.046557  [ 5064/ 5232]
Test Error: 
 Accuracy: 78.2%, Avg loss: 0.646443 

Epoch 2
-------------------------------
loss: 0.051527  [   24/ 5232]
loss: 0.148634  [  264/ 5232]
loss: 0.034600  [  504/ 5232]
loss: 0.025906  [  744/ 5232]
loss: 0.161083  [  984/ 5232]
loss: 0.031884  [ 1224/ 5232]
loss: 0.017559  [ 1464/ 5232

In [46]:
import torch
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for X, y in test_dataloader:
        X, y = X.to(device), y.to(device)
        pred = model(X)
        pred_labels = (pred.squeeze() > 0).float()   # sign of logit -> 0/1 prediction
        all_preds.extend(pred_labels.cpu().tolist())
        all_labels.extend(y.cpu().tolist())

In [47]:
cm = confusion_matrix(all_labels, all_preds)
print(cm)

print(classification_report(all_labels, all_preds, target_names=["NORMAL", "PNEUMONIA"]))

[[141  93]
 [ 28 362]]
              precision    recall  f1-score   support

      NORMAL       0.83      0.60      0.70       234
   PNEUMONIA       0.80      0.93      0.86       390

    accuracy                           0.81       624
   macro avg       0.81      0.77      0.78       624
weighted avg       0.81      0.81      0.80       624



In [57]:
torch.save(model.state_dict(), "PneumoNetV1WithFullyConnectedLayersOnly.pt")